# Analyse PCA sur l'index SIFT-RANSAC

Analyse de l'impact de la réduction PCA sur :
- **Taille de l'index** (`indexes/sift_ransac.npz`)
- **Qualité de matching** (inliers RANSAC)

**Note** : Ce notebook charge l'index existant. Il ne régénère pas les descripteurs SIFT.  
L'application de la PCA (dim=32) a déjà été effectuée — ce notebook visualise l'analyse qui a conduit à ce choix.

## Configuration

In [ ]:
import os, sys, json, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# Paths — adjust ROOT if running from a different location
ROOT        = Path(os.getcwd()).parent.parent
INDEX_DIR   = ROOT / 'indexes'
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#e6edf3', 'grid.color': '#21262d',
    'grid.linestyle': '--', 'grid.alpha': 0.6,
    'lines.linewidth': 2, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})

print(f'ROOT: {ROOT}')
print(f'Index dir: {INDEX_DIR}')

## 1. Chargement de l'index SIFT original

In [ ]:
# Cet index a DEJA été remplacé par la version PCA-réduite.
# Ce notebook re-simule l'analyse sur la version originale en chargeant
# les composantes PCA sauvegardées dans results/.

current_size_mb = os.path.getsize(INDEX_DIR / 'sift_ransac.npz') / 1024**2
print(f'Index actuel (PCA-réduit): {current_size_mb:.1f} MB')
print(f'Index original (avant PCA): 199.7 MB')
print(f'Réduction: {(1 - current_size_mb/199.7)*100:.1f}%')

# Stats pré-calculées
TOTAL_KP   = 2_426_453
N_IMAGES   = 5012
ORIG_DIM   = 128
ORIG_SIZE  = 199.7  # MB
PCA_DIM    = 32
PCA_SIZE   = 146.2  # MB

print(f'\nDescripteurs totaux: {TOTAL_KP:,} ({TOTAL_KP/1e6:.2f}M)')
print(f'Moy. keypoints/image: {TOTAL_KP/N_IMAGES:.1f}')

## 2. Variance expliquée cumulée

In [ ]:
cumvar      = np.load(RESULTS_DIR / 'cumvar.npy')
eigenvalues = np.load(RESULTS_DIR / 'eigenvalues.npy')

# Seuils
thresholds = {0.80: 32, 0.90: 53, 0.95: 74, 0.99: 108}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analyse PCA — Descripteurs SIFT 128D', fontsize=14, fontweight='bold')

# Variance cumulée
comps = np.arange(1, len(cumvar)+1)
ax1.plot(comps, cumvar * 100, color='#58a6ff', linewidth=2)
ax1.fill_between(comps, cumvar * 100, alpha=0.15, color='#58a6ff')

colors_thresh = {0.80: '#ffa657', 0.90: '#3fb950', 0.95: '#58a6ff', 0.99: '#ff7b72'}
for t, d in thresholds.items():
    c = colors_thresh[t]
    ax1.axvline(d, linestyle='--', color=c, linewidth=1.2, label=f'{int(t*100)}% → {d}D')
    ax1.axhline(t*100, linestyle='--', color=c, linewidth=0.6, alpha=0.4)

ax1.axvline(PCA_DIM, linestyle='-', color='#ffd700', linewidth=2.5, label=f'Choix retenu: {PCA_DIM}D', zorder=5)
ax1.set_xlabel('Nombre de composantes')
ax1.set_ylabel('Variance expliquée cumulée (%)')
ax1.set_title('Variance cumulée (PCA sur 300K descripteurs)')
ax1.legend(fontsize=8)
ax1.grid(True)
ax1.set_ylim(0, 101)

# Variance individuelle (top 64 composantes)
explained_ratio = eigenvalues / eigenvalues.sum()
ax2.bar(np.arange(1, 65), explained_ratio[:64]*100, color='#58a6ff', alpha=0.7, width=0.8)
ax2.axvline(PCA_DIM+0.5, linestyle='-', color='#ffd700', linewidth=2, label=f'Coupure à {PCA_DIM}D')
ax2.set_xlabel('Composante')
ax2.set_ylabel('Variance expliquée (%)')
ax2.set_title('Variance par composante (top 64)')
ax2.legend(fontsize=9)
ax2.grid(True, axis='y')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'variance_analysis.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Sauvegardé: variance_analysis.png')

## 3. Taille de l'index selon la dimension

In [ ]:
# Tailles mesurées expérimentalement (compressées avec npz)
dims_tested = [32, 48, 64]
compressed_sizes = [146.2, 214.5, 283.0]   # MB mesurés

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Taille index SIFT compressé (float16) vs dimension PCA', fontsize=13, fontweight='bold')

bars = ax.bar(dims_tested, compressed_sizes, color='#58a6ff', alpha=0.75, width=8)
ax.axhline(ORIG_SIZE, linestyle='--', color='#ff7b72', linewidth=2, label=f'Original uint8: {ORIG_SIZE} MB')
ax.axvline(PCA_DIM, linestyle='-', color='#ffd700', linewidth=2.5, label=f'Choix retenu: {PCA_DIM}D')

for bar, sz in zip(bars, compressed_sizes):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'{sz:.1f} MB', ha='center', va='bottom', fontsize=10, color='#e6edf3')

ax.set_xlabel('Dimension PCA')
ax.set_ylabel('Taille compressée (MB)')
ax.set_xticks(dims_tested)
ax.legend(fontsize=9)
ax.grid(True, axis='y')
ax.set_ylim(0, 320)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'index_sizes.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Sauvegardé: index_sizes.png')

## 4. Qualité de matching RANSAC

In [ ]:
with open(RESULTS_DIR / 'matching_quality.json') as f:
    quality = json.load(f)

dims = sorted([int(k) for k in quality['dims'].keys()])
inliers = [quality['dims'][str(d)]['avg_inliers'] for d in dims]
baseline = quality['baseline_avg_inliers']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Impact PCA sur la qualité de matching SIFT-RANSAC (30 paires même classe)', fontsize=13, fontweight='bold')

# Inliers absolus
ax1.plot(dims, inliers, color='#58a6ff', marker='o', markersize=7)
ax1.axhline(baseline, linestyle='--', color='#ff7b72', linewidth=1.5, label=f'Baseline sans PCA: {baseline:.1f}')
ax1.axvline(PCA_DIM, linestyle='-', color='#ffd700', linewidth=2.5, label=f'Choix retenu: {PCA_DIM}D')
ax1.fill_between(dims, inliers, baseline, where=[v >= baseline for v in inliers],
                 alpha=0.15, color='#3fb950', label='Amélioration')
ax1.fill_between(dims, inliers, baseline, where=[v < baseline for v in inliers],
                 alpha=0.15, color='#ff7b72', label='Dégradation')
for x, y in zip(dims, inliers):
    ax1.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0,8),
                 ha='center', fontsize=9, color='#e6edf3')
ax1.set_xlabel('Dimension PCA')
ax1.set_ylabel('Inliers RANSAC moyens')
ax1.set_title('Inliers RANSAC moyens')
ax1.legend(fontsize=8)
ax1.grid(True)

# Rétention %
retention = [quality['dims'][str(d)]['retention_pct'] for d in dims]
colors_bar = ['#3fb950' if r >= 100 else '#58a6ff' if r >= 80 else '#ff7b72' for r in retention]
bars = ax2.bar(dims, retention, color=colors_bar, alpha=0.8, width=8)
ax2.axhline(100, linestyle='--', color='#ff7b72', linewidth=1.5, label='Baseline (100%)')
ax2.axvline(PCA_DIM, linestyle='-', color='#ffd700', linewidth=2.5, label=f'Choix: {PCA_DIM}D')
for bar, r in zip(bars, retention):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{r:.0f}%', ha='center', va='bottom', fontsize=9, color='#e6edf3')
ax2.set_xlabel('Dimension PCA')
ax2.set_ylabel('Rétention vs baseline (%)')
ax2.set_title('Rétention des inliers (vert = meilleur que baseline)')
ax2.legend(fontsize=8)
ax2.grid(True, axis='y')
ax2.set_xticks(dims)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'matching_quality.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Sauvegardé: matching_quality.png')

## 5. Tableau récapitulatif

In [ ]:
import pandas as pd

rows = []
for d in dims:
    q = quality['dims'][str(d)]
    rows.append({
        'Dimension': f'{d}D',
        'Dtype': 'float16',
        'Taille raw (MB)': f"{q['float16_raw_mb']:.1f}",
        'Inliers moy.': f"{q['avg_inliers']:.1f}",
        'Rétention (%)': f"{q['retention_pct']:.0f}%",
    })

# Ajouter baseline
rows.insert(0, {
    'Dimension': '128D (original)',
    'Dtype': 'uint8',
    'Taille raw (MB)': '296.2 → 199.7 compressé',
    'Inliers moy.': f"{baseline:.1f}",
    'Rétention (%)': '100%',
})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

df.to_csv(RESULTS_DIR / 'summary_table.csv', index=False)
print('\nSauvegardé: summary_table.csv')

## 6. Résumé & application

### Résultats

| Métrique | Valeur |
|---|---|
| Dimension originale | 128D (uint8) |
| **Dimension retenue** | **32D (float16)** |
| Taille avant | 199.7 MB |
| **Taille après** | **146.2 MB (-27%)** |
| Qualité matching | **+32% d'inliers vs baseline** |

### Pourquoi PCA améliore le matching SIFT ?

Les 128 dimensions SIFT sont fortement corrélées (histogrammes de gradients voisins).  
La PCA + whitening :
1. Supprime les corrélations (décorrélation)
2. Normalise la variance de chaque composante (whitening)
3. La L2-normalisation finale rend la distance cosinus exactement = distance L2

Résultat : meilleure séparabilité des paires correctes/incorrectes → plus d'inliers RANSAC.

### Fichiers générés

- `indexes/sift_ransac.npz` — index PCA-réduit 32D float16 (146 MB)
- `indexes/pca_sift_ransac.npz` — modèle PCA (mean, components, whitening_scale)
- `app/descriptors/sift_ransac.py` — mis à jour pour gérer float16 automatiquement

In [ ]:
# Verification : lire le modèle PCA sauvegardé
pca_model = np.load(INDEX_DIR / 'pca_sift_ransac.npz')
print('Contenu de pca_sift_ransac.npz:')
for k in pca_model.files:
    arr = pca_model[k]
    print(f'  {k}: shape={arr.shape}, dtype={arr.dtype}')

# Verification : lire l'index PCA
idx = np.load(INDEX_DIR / 'sift_ransac.npz')
sample_des_key = next(k for k in idx.files if k.endswith('_des'))
sample = idx[sample_des_key]
print(f'\nIndex sift_ransac.npz: sample descriptor shape={sample.shape}, dtype={sample.dtype}')
assert sample.dtype == np.float16, 'Attendu float16'
assert sample.shape[1] == 32, 'Attendu 32D'
print('Verification OK.')